# LLM-Guided Chip Placement — Kaggle Runner

Mode A (training-free greedy) on the MaskPlace engine. **Setup pulls the code+data
straight from GitHub**, so you never re-upload a dataset — just `git push` and re-run.

gym 0.21.0 can't build on Py3.12, so we install a tiny in-memory `gym` shim
(Mode A only needs `gym.Env` + `spaces.Discrete`). No real gym, no torch.

**Run top to bottom.** Phase 2 baseline = Cell `Greedy (Mode B)`.

## Cell 1 — Setup: clone repo + gym shim + protobuf

In [ ]:
import subprocess, sys, os, shutil, types, glob
import numpy as np

REPO_URL  = "https://github.com/dennis5727/chipPlacement.git"
CLONE_DIR = "/kaggle/working/chipPlacement"
WORK_DIR  = os.path.join(CLONE_DIR, "maskplace")   # inner dir: code + ariane/ data

# ---- 1. Get code+data: prefer git (always latest); fall back to a Kaggle dataset ----
if not os.path.exists(CLONE_DIR):
    try:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR])
        print("cloned ->", CLONE_DIR)
    except Exception as e:
        print("git clone failed (private repo? no internet?):", e)
else:
    subprocess.call(["git", "-C", CLONE_DIR, "pull", "--ff-only"])
    print("repo pulled (latest)")

if not os.path.exists(WORK_DIR):                    # dataset fallback
    hits = glob.glob("/kaggle/input/**/place_db.py", recursive=True)
    assert hits, "No code found via git OR dataset. Turn Internet ON, or attach the dataset."
    shutil.copytree(os.path.dirname(hits[0]), WORK_DIR)
    print("used dataset fallback ->", WORK_DIR)

os.chdir(WORK_DIR)                                  # PlaceDB reads ariane/netlist.pb.txt relative to cwd
sys.path.insert(0, WORK_DIR)
print("cwd:", os.getcwd())

# ---- 2. protobuf for the generated laiyao_pb2.py ----
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "protobuf==3.20.3"])
except subprocess.CalledProcessError as e:
    print("protobuf pin failed; relying on pure-python impl:", e)

# ---- 3. gym shim (gym 0.21.0 won't build on Py3.12; Mode A only needs Env + Discrete) ----
try:
    import gym  # noqa: F401
    print("real gym:", gym.__version__)
except Exception:
    gym = types.ModuleType("gym"); spaces = types.ModuleType("gym.spaces")
    class _Env: pass
    class _Discrete:
        def __init__(self, n): self.n = int(n)
        def contains(self, x):
            try: x = int(x)
            except (TypeError, ValueError): return False
            return 0 <= x < self.n
    class _Box:
        def __init__(self, low=0, high=1, shape=None, dtype=None):
            self.low, self.high, self.shape, self.dtype = low, high, shape, dtype
    gym.Env = _Env; spaces.Discrete = _Discrete; spaces.Box = _Box; gym.spaces = spaces
    sys.modules["gym"] = gym; sys.modules["gym.spaces"] = spaces
    print("installed gym shim")

# ---- 4. (optional) LLM SDK for Phase 4+ ----
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anthropic"])
except subprocess.CalledProcessError as e:
    print("anthropic install skipped:", e)

for f in ["place_db.py","place_env/place_env.py","comp_res.py","greedy_place.py","ariane/netlist.pb.txt"]:
    assert os.path.exists(f), f"MISSING: {f}"
print("files OK\n=== SETUP COMPLETE ===")

## Cell 2 — Sanity check the engine

In [ ]:
from place_db import PlaceDB
placedb = PlaceDB("ariane")
hard = sum(1 for n in placedb.node_info if placedb.node_info[n].get("is_hard"))
print("Nodes:", len(placedb.node_info), "| Nets:", len(placedb.net_info),
      "| Canvas:", placedb.max_height, "| Hard MACROs:", hard)
assert placedb.max_height == 357

## Cell 3 — Greedy baseline (Mode B: 133 hard SRAM macros only)

This is **baseline row #1**. Done-when: ~133 placed, **0 overlaps**, finite HPWL.
If you see `no legal cell` / placed < 133, bump `grid=224` to `grid=300` (finer = easier packing).

In [ ]:
import importlib, greedy_place
importlib.reload(greedy_place)
from greedy_place import greedy_place as run_place

res = run_place(benchmark="ariane", hard_only=True, hard_order="keep",
                grid=224, save_fig="greedy_hard133.png", verbose=True)
print(f"\nMode B  HPWL: {res['hpwl']:.6e} | overlaps: {res['overlaps']} | placed: {res['placed']}")

## Cell 4 — Show the layout

In [ ]:
from IPython.display import Image, display
display(Image(filename="greedy_hard133.png"))